# RAG Workshop

## Resumenes de casos

In [34]:
%pip install google-genai dotenv sentence-transformers numpy faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.2 MB/s eta 0:00:00m eta 0:00:010:01:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from google import genai
from dotenv import load_dotenv
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

load_dotenv("./.env")
client = genai.Client()

/home/dani22002/repos/rag-workshop-devopsdays/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
for model in client.models.list():
    if "flash" in model.name.lower():
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [29]:
interaction = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Explain Kubernetes network policies in three short sentences.",
)

In [30]:
print(interaction.text)

Kubernetes network policies are security rules that control how pods communicate with each other and other network endpoints. They act as a virtual firewall, using labels and selectors to specify which traffic is allowed or denied. By default, pods are non-isolated, but these policies enable a zero-trust model by restricting access to authorized connections only.


In [ ]:
MODEL_NAME = "BAAI/bge-base-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

case_texts = [
    "Example 1",
    "Example 2"
]

case_vectors = model.encode(
    case_texts,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

In [ ]:
index = faiss.IndexFlatIP(case_vectors.shape[1])  # Inner Product (cosine similarity if vectors are normalized)
index.add(case_vectors)

In [ ]:
query_vector = model.encode(
    "Represent this sentence for searching relevant passages: " + query,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

# FAISS expects a two-dimensional array: (number_of_queries, dimension).
query_vector = query_vector.reshape(1, -1)

In [ ]:
top_k = 3
scores, positions = index.search(query_vector, top_k)

In [ ]:
faiss.write_index(index, "openshift_cases.faiss")

In [ ]:
index = faiss.read_index("openshift_cases.faiss")